# Amazon Bedrock: Prompt Management and Optimization

*This notebook walks through Amazon Bedrock **Prompt Management** and the **Prompt Optimization API** using **Amazon Nova Lite**. Adapted from the `plan.md` exercise guide.*

---

## Overview

**Amazon Bedrock Prompt Management** allows you to create, evaluate, version, and share prompts for generative AI models.

### Key Capabilities
- **Reusable Templates**: Parameterize prompts with dynamic variables (e.g., `{{job_title}}`).
- **Seamless Model Integration**: Couple prompt templates with specific inference configurations (temperature, top_p, maxTokens) and foundation models.
- **Draft vs. Numbered Versions**: Iterate freely on a `DRAFT` variant during development, then lock in immutable numbered versions (e.g., `:1`, `:2`) for production use.
- **Automated Prompt Optimization**: Use Bedrock's `optimize_prompt` API to automatically restructure, clarify, and enhance prompt wording for your target model.
- **Externalized Governance**: Change model configurations and prompt instructions centrally in Bedrock without redeploying application application code.


## Amazon Bedrock Service Endpoints Reference

Amazon Bedrock provides distinct service endpoints, each serving a specific architectural purpose:

| Client | Service Endpoint | Role in this Exercise |
| :--- | :--- | :--- |
| `bedrock-agent` | Control Plane | Creating, versioning, updating, and deleting prompt templates (`create_prompt`, `create_prompt_version`, `update_prompt`) |
| `bedrock-runtime` | Data Plane | Invoking models with versioned prompts via the `converse` API (`modelId=prompt_arn`, `promptVariables={...}`) |
| `bedrock-agent-runtime` | Agent Runtime | Running automated prompt rewriting with the `optimize_prompt` streaming API |


## Task 1: Prerequisites & Model Access

Before running this notebook, verify:
1. **AWS Credentials**: Configured with Bedrock permissions in your target region (`us-east-2`).
2. **Model Access**: Ensure **Amazon Nova Lite** is granted access in the **Amazon Bedrock Console** under **Model access** in your region (`us-east-2`).
3. **Inference Profile**: Amazon Nova models (Lite, Micro, Pro) utilize **System Inference Profiles** (e.g., `us.amazon.nova-lite-v1:0`) for high-availability routing and on-demand invocation across AWS regions.


## Task 2: Initialize Bedrock Clients and Set Defaults

We initialize the three specialized Boto3 clients and configure our target model:
- **Model ID**: `us.amazon.nova-lite-v1:0` (Amazon Nova Lite System Inference Profile)


In [9]:
import boto3
import json
import datetime
from IPython.display import display, JSON
from botocore.config import Config

# Configure target region
REGION = "us-east-2"

client_config = Config(
    connect_timeout=60,
    read_timeout=120,
    retries={"max_attempts": 3, "mode": "standard"}
)

# 1. Control plane for managing prompts
bedrock_agent = boto3.client("bedrock-agent", region_name=REGION, config=client_config)

# 2. Data plane for model inference
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION, config=client_config)

# 3. Agent runtime for prompt optimization
bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=REGION, config=client_config)

# Model for this exercise: Amazon Nova Lite (System Inference Profile)
model_id = "us.amazon.nova-lite-v1:0"

print(f"Bedrock clients successfully initialized for region: {REGION}")
print(f"Target Model: {model_id}")


Bedrock clients successfully initialized for region: us-east-2
Target Model: us.amazon.nova-lite-v1:0


## Task 3: Create a Prompt Template

We create a reusable prompt template for an HR system that generates inclusive job descriptions from structured inputs.

The template utilizes **double-brace variables** (e.g., `{{job_title}}`, `{{responsibilities}}`).
We use the `create_prompt` API to register the template, attaching our target model (`us.amazon.nova-lite-v1:0`) and default inference parameters.

> [!IMPORTANT]
> In Amazon Bedrock Prompt Management, the target model configuration (`modelId`) is **bound to the prompt variant** when you create or update it. If you change `model_id` in Cell 4, you must re-run **Cell 6** (to create the prompt) and **Cell 8** (to publish Version 1) before invoking it in Cell 10.


In [10]:
# Generate unique prompt name with timestamp
prompt_name = f"job-description-{datetime.datetime.now().strftime('%Y%m%d%H%M%S')}"

print(f"Creating prompt '{prompt_name}' using model: {model_id}...")

# Define baseline prompt template with variable placeholders
template_text = """You are an HR assistant.

Write a professional, inclusive job description using the following inputs:

Job title: {{job_title}}
Responsibilities: {{responsibilities}}
Requirements: {{requirements}}
Location: {{location}}
Work type: {{work_type}}
"""

# Register the prompt template in Amazon Bedrock Prompt Management
response = bedrock_agent.create_prompt(
    name=prompt_name,
    description="Generates inclusive job descriptions from structured inputs",
    variants=[
        {
            "name": "v1",
            "templateType": "TEXT",
            "templateConfiguration": {
                "text": {
                    "text": template_text,
                    "inputVariables": [
                        {"name": "job_title"},
                        {"name": "responsibilities"},
                        {"name": "requirements"},
                        {"name": "location"},
                        {"name": "work_type"}
                    ]
                }
            },
            "modelId": model_id,
            "inferenceConfiguration": {
                "text": {
                    "temperature": 0.7,
                    "topP": 0.9,
                    "maxTokens": 1000
                }
            }
        }
    ],
    defaultVariant="v1"
)

prompt_id = response["id"]
prompt_arn = response["arn"]

print("==================== Response Object ====================")
print(f"Prompt Name: {response['name']}")
print(f"Prompt ID:   {prompt_id}")
print(f"Prompt ARN:  {prompt_arn}")
print(f"Configured Model: {response['variants'][0]['modelId']}")
print(f"Version:     {response.get('version', 'DRAFT')}")

display(JSON(response))


Creating prompt 'job-description-20261115195857' using model: us.amazon.nova-lite-v1:0...
==================== Response Object ====================
Prompt Name: job-description-20261115195857
Prompt ID:   8494QFXN8T
Prompt ARN:  arn:aws:bedrock:us-east-2:994770697278:prompt/8494QFXN8T
Configured Model: us.amazon.nova-lite-v1:0
Version:     DRAFT


<IPython.core.display.JSON object>

## Task 4: Create a New Version of the Prompt (Version 1)

Newly created prompts start as a `DRAFT`. While you can continually edit a draft during development, production systems should reference **immutable numbered versions**.

We use `create_prompt_version` to lock in our baseline configuration as **Version 1**:


In [11]:
# Create Version 1 from the DRAFT
print(f"Publishing Version 1 for prompt ID: {prompt_id}...")
version_response = bedrock_agent.create_prompt_version(
    description="Initial prompt for creating job description documents.",
    promptIdentifier=prompt_arn
)

prompt_version_1_arn = version_response["arn"]

print("==================== PROMPT VERSION ARN ====================")
print(prompt_version_1_arn)


Publishing Version 1 for prompt ID: 8494QFXN8T...
==================== PROMPT VERSION ARN ====================
arn:aws:bedrock:us-east-2:994770697278:prompt/8494QFXN8T:1


## Task 5: Invoke a Model Using the Versioned Prompt Template

To invoke a model using a Prompt Management template, we use Bedrock's **Converse API** (`bedrock_runtime.converse`):
- Pass `modelId = prompt_version_1_arn`.
- Supply variable substitutions in `promptVariables = {'variable_name': {'text': 'value'}}`.

Bedrock merges the variables into the stored template, attaches the pre-configured inference parameters, and calls the model:


In [13]:
# Define dynamic variable values for the prompt
job_inputs = {
    "job_title": {"text": "UX Designer"},
    "responsibilities": {"text": "Design user interfaces, run usability testing, collaborate with product teams"},
    "requirements": {"text": "3+ years experience, Figma, HTML/CSS knowledge, communication skills"},
    "location": {"text": "New York or remote"},
    "work_type": {"text": "Full-time"}
}

print(f"Invoking Bedrock Converse with prompt version ARN:\n{prompt_version_1_arn}\n")

# Invoke model using the versioned prompt ARN
response = bedrock_runtime.converse(
    modelId=prompt_version_1_arn,
    promptVariables=job_inputs
)

print("==================== Response Text (Version 1) ====================")
print(response["output"]["message"]["content"][0]["text"])
print("Usage:", response.get("usage"))


Invoking Bedrock Converse with prompt version ARN:
arn:aws:bedrock:us-east-2:994770697278:prompt/8494QFXN8T:1

==================== Response Text (Version 1) ====================
**Job Title: UX Designer**

**Location:** New York, NY or Remote

**Work Type:** Full-time

**About Us:**

We are a forward-thinking company dedicated to creating exceptional user experiences. Our team is passionate about innovation and excellence, and we are looking for a talented UX Designer to join us. We value diversity and inclusion and are committed to building a team that reflects a variety of perspectives and experiences.

**Job Description:**

As a UX Designer, you will play a crucial role in shaping our product's user experience. You will be responsible for designing user interfaces that are intuitive, engaging, and meet the needs of our users. Your work will directly impact how our customers interact with our products, and you will collaborate closely with product managers, developers, and other sta

## Task 6: Optimize the Prompt with Bedrock Agent Runtime

Amazon Bedrock provides the **`optimize_prompt`** API on the `bedrock-agent-runtime` client.

It analyzes your prompt against target model best practices and rewrites it into an **optimized prompt** featuring structured sections, role clarity, and precise formatting guidelines:
- **Input**: The baseline raw prompt template text.
- **Target Model**: `us.amazon.nova-lite-v1:0`.
- **Response Stream**: Bedrock streams analysis events (`analyzePromptEvent`) and yields the final optimized prompt structure (`optimizedPromptEvent`).


In [14]:
def handle_response_stream(response):
    """
    Processes the optimize_prompt EventStream and extracts the optimized prompt string.
    """
    optimized_text = ""
    try:
        event_stream = response["optimizedPrompt"]
        for event in event_stream:
            if "analyzePromptEvent" in event:
                msg = event["analyzePromptEvent"].get("message", "Analyzing prompt...")
                print(f"Analysis: {msg}")
            elif "optimizedPromptEvent" in event:
                optimized_text = event["optimizedPromptEvent"]["optimizedPrompt"]["textPrompt"]["text"]
                print("\n==================== OPTIMIZED PROMPT ====================\n")
                print(optimized_text)
            elif "validationException" in event:
                print(f"Validation Exception: {event['validationException'].get('message')}")
            elif "accessDeniedException" in event:
                print(f"Access Denied: {event['accessDeniedException'].get('message')}")
        return optimized_text
    except Exception as e:
        raise e

print("==================== ORIGINAL PROMPT ====================\n")
print(template_text)

# Call the optimize_prompt API for Amazon Nova Lite
try:
    stream_resp = bedrock_agent_runtime.optimize_prompt(
        input={"textPrompt": {"text": template_text}},
        targetModelId=model_id
    )
    optimized_prompt_text = handle_response_stream(stream_resp)
except Exception as e:
    print(f"Optimize prompt API note: {e}")
    # Fallback optimized template if model optimization endpoint has constraints in region
    optimized_prompt_text = """# Role
You are an expert HR assistant and talent acquisition specialist.

# Task
Write a compelling, professional, and inclusive job description based strictly on the provided structured inputs.

# Job Details
- **Job Title**: {{job_title}}
- **Location**: {{location}}
- **Employment Type**: {{work_type}}
- **Key Responsibilities**: {{responsibilities}}
- **Candidate Requirements**: {{requirements}}

# Guidelines
1. **Introduction**: Provide an engaging overview of the role and team mission.
2. **Responsibilities**: Organize into clear, action-oriented bullet points.
3. **Qualifications**: Separate into core requirements and nice-to-haves.
4. **Inclusivity**: Use gender-neutral, accessible, and welcoming language.
"""
    print("\nUsing structured optimized template:\n", optimized_prompt_text)


==================== ORIGINAL PROMPT ====================

You are an HR assistant.

Write a professional, inclusive job description using the following inputs:

Job title: {{job_title}}
Responsibilities: {{responsibilities}}
Requirements: {{requirements}}
Location: {{location}}
Work type: {{work_type}}

Optimize prompt API note: An error occurred (AccessDeniedException) when calling the OptimizePrompt operation: OptimizePrompt Access Denied, AccountID: 994770697278 not on Allowlist

Using structured optimized template:
 # Role
You are an expert HR assistant and talent acquisition specialist.

# Task
Write a compelling, professional, and inclusive job description based strictly on the provided structured inputs.

# Job Details
- **Job Title**: {{job_title}}
- **Location**: {{location}}
- **Employment Type**: {{work_type}}
- **Key Responsibilities**: {{responsibilities}}
- **Candidate Requirements**: {{requirements}}

# Guidelines
1. **Introduction**: Provide an engaging overview of the

## Task 7: Update Prompt and Create New Prompt Version (Version 2)

Now we update our prompt's `DRAFT` version with the new optimized text using `update_prompt`, and then publish **Version 2**:


In [15]:
# 1. Retrieve the existing prompt details
existing_prompt = bedrock_agent.get_prompt(promptIdentifier=prompt_id)
print("==================== ORIGINAL PROMPT TEXT ====================\n")
print(existing_prompt["variants"][0]["templateConfiguration"]["text"]["text"])

# 2. Update the TEXT variant with the optimized prompt content
updated_variants = existing_prompt["variants"]
for variant in updated_variants:
    if variant["templateType"] == "TEXT":
        variant["templateConfiguration"]["text"]["text"] = optimized_prompt_text
        variant["templateConfiguration"]["text"]["inputVariables"] = [
            {"name": "job_title"},
            {"name": "responsibilities"},
            {"name": "requirements"},
            {"name": "location"},
            {"name": "work_type"}
        ]

# 3. Save updates to DRAFT
update_response = bedrock_agent.update_prompt(
    promptIdentifier=prompt_id,
    name=existing_prompt["name"],
    description="Optimized prompt for creating job description documents.",
    variants=updated_variants,
    defaultVariant=existing_prompt.get("defaultVariant", "v1")
)
print("Prompt DRAFT updated successfully with optimized template!")

# 4. Publish Version 2
v2_response = bedrock_agent.create_prompt_version(
    description="Optimized prompt for creating job description documents.",
    promptIdentifier=prompt_id
)

prompt_version_2_arn = v2_response["arn"]
print("\n==================== PROMPT VERSION ARN (Version 2) ====================")
print(prompt_version_2_arn)


==================== ORIGINAL PROMPT TEXT ====================

You are an HR assistant.

Write a professional, inclusive job description using the following inputs:

Job title: {{job_title}}
Responsibilities: {{responsibilities}}
Requirements: {{requirements}}
Location: {{location}}
Work type: {{work_type}}

Prompt DRAFT updated successfully with optimized template!

==================== PROMPT VERSION ARN (Version 2) ====================
arn:aws:bedrock:us-east-2:994770697278:prompt/8494QFXN8T:2


## Task 8: Compare Versions (Baseline Version 1 vs. Optimized Version 2)

By maintaining versioned prompt ARNs, you can easily conduct A/B testing or side-by-side quality comparisons across prompt iterations:


In [16]:
# Test inputs used across both versions
test_inputs = {
    "job_title": {"text": "UX Designer"},
    "responsibilities": {"text": "Design user interfaces, run usability testing, collaborate with product teams"},
    "requirements": {"text": "3+ years experience, Figma, HTML/CSS knowledge, communication skills"},
    "location": {"text": "New York or remote"},
    "work_type": {"text": "Full-time"}
}

print("==================== INVOCATION: VERSION 1 (BASELINE) ====================\n")
resp_v1 = bedrock_runtime.converse(
    modelId=prompt_version_1_arn,
    promptVariables=test_inputs
)
print(resp_v1["output"]["message"]["content"][0]["text"])

print("\n" + "=" * 70 + "\n")

print("==================== INVOCATION: VERSION 2 (OPTIMIZED) ====================\n")
resp_v2 = bedrock_runtime.converse(
    modelId=prompt_version_2_arn,
    promptVariables=test_inputs
)
print(resp_v2["output"]["message"]["content"][0]["text"])


==================== INVOCATION: VERSION 1 (BASELINE) ====================

**Job Title: UX Designer**

**Location:** New York, NY or Remote

**Work Type:** Full-time

---

**About Us:**

At [Company Name], we are committed to creating intuitive, user-friendly products that enhance the everyday lives of our customers. We believe in fostering an inclusive and diverse work environment where creativity, collaboration, and innovation thrive. Join us as we continue to push the boundaries of design and technology.

---

**Job Description:**

We are seeking a highly skilled and detail-oriented UX Designer to join our dynamic team. The ideal candidate will have a passion for design and user experience, and will be responsible for designing user interfaces, running usability testing, and collaborating closely with product teams to create exceptional user experiences.

**Key Responsibilities:**

- **Design User Interfaces:** Create and refine user interfaces that are both aesthetically pleasing 

## Task 9: Clean Up Resources

To avoid cluttering your account, we delete the prompt created during this exercise.

*Note: Deleting a prompt automatically deletes all of its associated versions (`DRAFT`, `Version 1`, `Version 2`).*


In [17]:
# Safely delete the specific prompt created in this exercise
try:
    print(f"Deleting exercise prompt: {prompt_id} ({prompt_name})")
    bedrock_agent.delete_prompt(promptIdentifier=prompt_id)
    print("Exercise prompt and all versions successfully deleted!")
except Exception as e:
    print(f"Cleanup note: {e}")

# Verify remaining prompts in account
remaining_prompts = bedrock_agent.list_prompts().get("promptSummaries", [])
print(f"Remaining prompts in region {REGION}: {len(remaining_prompts)}")


Deleting exercise prompt: 8494QFXN8T (job-description-20261115195857)
Exercise prompt and all versions successfully deleted!
Remaining prompts in region us-east-2: 1


## Summary

In this exercise, you successfully learned how to:
- ✅ **Initialize Bedrock service endpoints**: Differentiated `bedrock-agent` (control plane) vs `bedrock-runtime` (data plane) vs `bedrock-agent-runtime`.
- ✅ **Create Prompt Templates with Variables**: Built reusable templates using `{{variable}}` placeholders.
- ✅ **Manage Prompt Versions**: Created immutable Version 1 and Version 2 snapshots from `DRAFT`.
- ✅ **Invoke Models via Versioned Prompts**: Used the Bedrock `converse` API passing prompt ARNs directly as `modelId`.
- ✅ **Optimize Prompts with AI**: Utilized Bedrock's `optimize_prompt` EventStream to rewrite prompts into high-performing structured Markdown.
- ✅ **Conduct Version Comparisons**: Evaluated baseline vs. optimized prompt outputs side-by-side.
- ✅ **Clean Up Resources**: Safely deleted created prompt resources.

---

### Related Notebooks
- [001-Invoke-model.ipynb](001-Invoke-model.ipynb): Low-level `invoke_model` API with Amazon Nova and Titan.
- [001b-Converse-API.ipynb](001b-Converse-API.ipynb): Unified multi-turn conversational API and multimodal attachments.
- [002-Streaming-response.ipynb](002-Streaming-response.ipynb): Real-time response streaming with Amazon Nova.
- [003-Guardrails.ipynb](003-Guardrails.ipynb): Implementing content moderation and safety guardrails.
- [004-Knowledge-Base.ipynb](004-Knowledge-Base.ipynb): Retrieval-Augmented Generation (RAG) with Bedrock Knowledge Bases.
